# Local Chat Debug Notebook

Type a user question in the **Ask** cell and run it to see every step the workflow took: planner decision, agents executed, SQL, verifier verdict, and final reply.

Run cells top-to-bottom once, then re-run only the **Ask** cell for each new question.

## 1. Setup — env + autoreload

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, asyncio, logging
from pathlib import Path
from dotenv import load_dotenv

# Ensure project root on sys.path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

# cd to project root so relative paths (e.g. bin/secrets/...) resolve.
os.chdir(ROOT)

# Resolve GCP credentials path to absolute (relative paths break when cwd changes).
_gcp = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
if _gcp and not os.path.isabs(_gcp):
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str((ROOT / _gcp).resolve())

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)

print('Root:', ROOT)
print('OPENAI_API_KEY set:', bool(os.getenv('OPENAI_API_KEY')))
print('GOOGLE_APPLICATION_CREDENTIALS:', os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Root: /Users/nelsoncamoes/dev/personal/worldcup_2026
OPENAI_API_KEY set: True
GOOGLE_APPLICATION_CREDENTIALS: /Users/nelsoncamoes/dev/personal/worldcup_2026/bin/secrets/gcp_service_account.json


## 2. Imports + helpers

In [5]:
from src.agents import orchestrator as orch
from src.agents.workflow_logger import reset_tracker, get_tracker
from src.agents.conversation_memory import ConversationMemory

# Conversation history — survives across runs of the Ask cell so multi-turn works.
CONVERSATION: list[dict[str, str]] = []

def _fmt(value, max_len=600):
    try:
        s = json.dumps(value, indent=2, default=str, ensure_ascii=False)
    except Exception:
        s = str(value)
    return s if len(s) <= max_len else s[:max_len] + f'\n... [truncated {len(s) - max_len} chars]'

async def debug_ask(question: str, user_id: str = 'debug-user') -> dict:
    """Invoke the compiled graph directly so we can inspect full state."""
    reset_tracker()
    memory = ConversationMemory.from_messages(CONVERSATION)
    memory.append_user(question)
    state = {
        'user_id': user_id,
        'user_message': question,
        'conversation_context': memory.context_block(),
        'topic': '',
        'selected_agent': '',
        'selected_agents': [],
        'needs_verifier': False,
        'agent_outputs': {},
        'agent_payload': {},
        'verifier_verdict': {},
        'confidence_score': 0.0,
        'confidence_label': 'low',
        'confidence_reason': '',
        'final_reply': '',
        'messages': [],
    }
    final_state = await orch._graph.ainvoke(state)
    CONVERSATION.append({'role': 'user', 'content': question})
    CONVERSATION.append({'role': 'assistant', 'content': final_state['final_reply']})
    return final_state

def print_report(final_state: dict) -> None:
    tracker = get_tracker()
    print('═' * 80)
    print('USER:', final_state.get('user_message'))
    print('═' * 80)

    # 1. Plan
    print('\n── 1. PLAN ──────────────────────────────────────────')
    print(f"  topic           : {final_state.get('topic')}")
    print(f"  primary agent   : {final_state.get('selected_agent')}")
    print(f"  all agents      : {final_state.get('selected_agents')}")
    print(f"  needs_verifier  : {final_state.get('needs_verifier')}")

    # 2. Each agent output
    print('\n── 2. AGENT OUTPUTS ─────────────────────────────────')
    for name, out in (final_state.get('agent_outputs') or {}).items():
        print(f'\n  ▸ {name}')
        print(f"      confidence: {out.get('confidence')}")
        ans = (out.get('answer') or '')[:500]
        print(f"      answer    : {ans}")
        meta = out.get('metadata') or {}
        if meta:
            print('      metadata  :')
            for k, v in meta.items():
                preview = _fmt(v, 400).replace('\n', '\n        ')
                print(f'        {k}: {preview}')

    # 3. Verifier
    verdict = final_state.get('verifier_verdict') or {}
    print('\n── 3. VERIFIER ──────────────────────────────────────')
    if verdict:
        for k, v in verdict.items():
            print(f'  {k}: {_fmt(v, 300)}')
    else:
        print('  (skipped)')

    # 4. Confidence + final reply
    print('\n── 4. FINAL ─────────────────────────────────────────')
    print(f"  confidence: {final_state.get('confidence_label')} ({final_state.get('confidence_score'):.2f})")
    print(f"  reason    : {final_state.get('confidence_reason')}")
    print('\n  REPLY:')
    print('  ' + (final_state.get('final_reply') or '').replace('\n', '\n  '))

    # 5. Step trace
    print('\n── 5. WORKFLOW TRACE ────────────────────────────────')
    print(tracker.get_summary())

print('Helpers ready.')

Helpers ready.


## 3. Ask — edit `QUESTION` and run

Multi-turn: each run appends to `CONVERSATION`. Clear it via the cell below to start fresh.

In [3]:
QUESTION = "What are Mexico's next 5 games?"

final_state = await debug_ask(QUESTION)
print_report(final_state)

WARNING src.tools.entity_resolver: entity_resolver: failed to load dim_team: File bin/secrets/gcp_service_account.json was not found.


════════════════════════════════════════════════════════════════════════════════
USER: What are Mexico's next 5 games?
════════════════════════════════════════════════════════════════════════════════

── 1. PLAN ──────────────────────────────────────────
  topic           : Mexico schedule
  primary agent   : bigquery
  all agents      : ['bigquery']
  needs_verifier  : True

── 2. AGENT OUTPUTS ─────────────────────────────────

  ▸ bigquery
      confidence: None
      answer    : It seems that the team information for Mexico is currently unavailable, which prevents me from retrieving their upcoming matches. Please try again later or check if there is an issue with the data source.
      metadata  :
        data_source: "bigquery"
        tool_calls: [
          {
            "tool": "resolve_team",
            "args": {
              "name": "Mexico"
            }
          }
        ]
        sql_executed: []
        row_samples: []

── 3. VERIFIER ─────────────────────────────────

## 4. Inspect raw state / tracker (optional)

In [ ]:
# Full structured JSON of every node execution
print(get_tracker().get_json())

In [ ]:
# Full final state (huge — useful for deep dives)
print(_fmt(final_state, max_len=8000))

## 5. Utilities

In [ ]:
# Reset the conversation history (start a new chat)
CONVERSATION.clear()
print('Conversation cleared.')

In [ ]:
# Clear the entity resolver cache (force re-load from BigQuery)
from src.tools.entity_resolver import clear_cache
clear_cache()
print('Entity resolver cache cleared.')